# dialogue_6_dissonance.py

- Client = LLM

- Therapist = LLM dissonance-aware (เห็น text VA + speech VA + delta แบบออนไลน์)

## 1. OpenAI client

In [1]:
import os
import json
import getpass
from typing import Tuple
from pathlib import Path
from openai import OpenAI
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "gpt-4o-mini"   # เปลี่ยนได้

def setup_client() -> OpenAI:
    # บังคับถาม key ทุกครั้ง
    if "OPENAI_API_KEY" in os.environ:
        del os.environ["OPENAI_API_KEY"]
    key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = key
    return OpenAI()

client = setup_client()

## 2. Text VA: ใช้ vad-bert (เหมือน dialogue_5)

### Check device (cuda is needed for speed improvement)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
VAD_MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(VAD_MODEL_NAME)
vad_model = AutoModelForSequenceClassification.from_pretrained(VAD_MODEL_NAME).to(device)
vad_model.eval()

V_MIN, V_MAX = 1.0, 5.0
A_MIN, A_MAX = 1.0, 5.0

def _to_minus1_1(x: float, xmin: float = 1.0, xmax: float = 5.0) -> float:
    return float(2 * (x - xmin) / (xmax - xmin) - 1.0)

def get_text_VA(text: str) -> Tuple[float, float]:
    enc = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = vad_model(**enc)

    vad = out.logits.cpu().numpy()[0]  # [V, A, D]
    v_raw, a_raw, d_raw = vad.tolist()

    v_norm = _to_minus1_1(v_raw, V_MIN, V_MAX)
    a_norm = _to_minus1_1(a_raw, A_MIN, A_MAX)
    return v_norm, a_norm



## 3. Speech: synth + VA (Old)

In [4]:
# import torch
# import subprocess
# from pathlib import Path
# import soundfile as sf
# import librosa
# import numpy as np
# from transformers import AutoModelForAudioClassification

# from typing import Tuple

# VOICE_DIR = Path(r"C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\voice")
# SYNTH_SCRIPT = Path(r"C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\run_synthesis_dialogue_6.py")

# def synthesize_client_audio(text: str, turn: int) -> Path:
#     cmd = ["python", str(SYNTH_SCRIPT), "--idx", str(turn)]
#     # หรือถ้า script รองรับ text ด้วยก็เพิ่ม args ตรงนี้
#     subprocess.run(cmd, check=True)

#     wav_path = VOICE_DIR / f"dialogue_6_utterance_{turn}.wav"
#     if not wav_path.exists():
#         raise FileNotFoundError(f"Expected audio not found: {wav_path}")
#     return wav_path


# WAVLM_MODEL_NAME = "3loi/SER-Odyssey-Baseline-WavLM-Multi-Attributes"

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print(f"Loading WavLM emotion model {WAVLM_MODEL_NAME} on {device} ...")
# _wavlm = AutoModelForAudioClassification.from_pretrained(
#     WAVLM_MODEL_NAME,
#     trust_remote_code=True,
# ).to(device)
# _wavlm.eval()

# _target_sr = _wavlm.config.sampling_rate
# _mean = _wavlm.config.mean
# _std = _wavlm.config.std
# _id2label = _wavlm.config.id2label  # {0: 'arousal', 1: 'dominance', 2: 'valence'}
# print("WavLM id2label:", _id2label)


# def _predict_file(path: str) -> Tuple[float, float, float]:
#     """
#     คืนค่า (aro, dom, val) ช่วงประมาณ 0..1 จากไฟล์เสียงเดียว
#     """
#     audio, sr = sf.read(path)
#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)

#     if sr != _target_sr:
#         audio = librosa.resample(audio, orig_sr=sr, target_sr=_target_sr)
#         sr = _target_sr

#     audio = (audio - _mean) / (_std + 1e-6)

#     wavs = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(device)
#     mask = torch.ones(1, wavs.shape[1], dtype=torch.float32).to(device)

#     with torch.no_grad():
#         pred = _wavlm(wavs, mask)

#     logits = pred.cpu().numpy()[0].astype(float)  # [A, D, V]
#     aro = float(logits[0])
#     dom = float(logits[1])
#     val = float(logits[2])
#     return aro, dom, val


# def _scale_0_1_to_minus1_1(x: float) -> float:
#     # ถ้า model ให้ 0..1, map ไป -1..1
#     return 2.0 * x - 1.0


# def get_speech_VA(wav_path: Path) -> Tuple[float, float]:
#     """
#     รับ path ของ wav แล้วคืน (val_s, aro_s) ในช่วง [-1, 1]
#     """
#     aro, dom, val = _predict_file(str(wav_path))
#     val_s = _scale_0_1_to_minus1_1(val)
#     aro_s = _scale_0_1_to_minus1_1(aro)
#     return val_s, aro_s



## 3. Speech: Zonos (real-time) + WavLM VA

In [5]:
# Import synthesis function for zonos 
import sys
from pathlib import Path
import os

# ชี้ path ไปโฟลเดอร์ที่มี run_synthesis_dialogue_6-2.py
BASE_DIR = Path(r"C:\Luna-AI-Therapist")
SYNTH_DIR = BASE_DIR / "dissonance" / "own_script" / "dialogue_6"
sys.path.insert(0, str(SYNTH_DIR))

# import ฟังก์ชัน synth จากไฟล์นั้น
from run_synthesis_dialogue_6_module import synth_single_utterance

Zonos DEFAULT_DEVICE: cuda:0
Zonos device: cuda
Loading Zonos model once at import...
Loading Zonos model: Zonos-v0.1-transformer
Zonos model loaded.
Model SR: 44100
Zonos ready.


In [6]:
# ==============================
# 3) Speech: Zonos (real-time) + WavLM VA
# ==============================

import os
import re
import json
import subprocess
from pathlib import Path
from typing import Tuple

import torch
import soundfile as sf
import librosa
import numpy as np
from transformers import AutoModelForAudioClassification

# ---- paths ----
VOICE_DIR = Path(r"C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\voice")
SYNTH_SCRIPT = Path(r"C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\run_synthesis_dialogue_6.py")

# JSON ชั่วคราวต่อ 1 utterance (สำหรับ Zonos)
TMP_ZONOS_JSON = Path(r"C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\tmp_directed_zonos_single.json")


# ---------------------------------
# 3.1 Zonos director: text -> directed_utterance (1 utterance)
# ---------------------------------

ZONOS_DIRECTOR_SYSTEM = """
You are a master Vocal Director simulating the emotion2vec framework.

Given ONE client utterance from a CBT therapy session, you must output
a JSON object with a single directed utterance for Zonos, matching this schema:

{
  "utterance_text": "...",
  "is_new_utterance_rule": true,
  "utterance_level_direction": "[anxious, slow]",
  "new_utterance_rule_definition": {
    "primary_zonos_vector_value": {
      "Happiness": 0.0,
      "Sadness": 0.8,
      "Fear": 0.4
    },
    "speaking_rate": 15.0,
    "pitch_std": 100.0
  },
  "frame_level_directions": []
}

Rules:
- Copy the client utterance EXACTLY into "utterance_text".
  Do NOT rewrite, paraphrase, summarize, or change any words.
- Use only these emotion keys in primary_zonos_vector_value:
  Happiness, Sadness, Disgust, Fear, Surprise, Anger, Neutral, Other.
- Values should be between -1.0 and 1.0.
- speaking_rate: between 10.0 and 25.0
- pitch_std: between 20.0 and 150.0
- is_new_utterance_rule must always be true.
- frame_level_directions can be an empty list [].

Output ONLY the JSON object, with keys exactly:
utterance_text, is_new_utterance_rule, utterance_level_direction,
new_utterance_rule_definition, primary_zonos_vector_value,
speaking_rate, pitch_std, frame_level_directions.
Do NOT include any extra commentary.
"""


def make_directed_zonos_for_text(client_text: str) -> dict:
    """
    รับ client_text 1 utterance แล้วให้ LLM สร้าง directed_utterance
    ที่ schema เหมือน element ใน "directed_utterances" ของ dialogue_6_directed_zonos.json
    """
    user_prompt = f"""
Client utterance:

\"\"\"{client_text}\"\"\"

Generate ONE directed utterance JSON following the schema and rules.
Output only the JSON.
"""
    raw = chat_once(ZONOS_DIRECTOR_SYSTEM, user_prompt)

    # ดึง JSON ก้อนแรกออกมาแบบหยาบ ๆ
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        raise ValueError(f"Could not find JSON in Zonos director output:\n{raw}")

    directed = json.loads(m.group(0))
    return directed


def write_tmp_zonos_json(directed: dict) -> None:
    """
    เขียน JSON ชั่วคราวสำหรับ Zonos:
    { "directed_utterances": [ directed ] }
    """
    data = {"directed_utterances": [directed]}
    TMP_ZONOS_JSON.parent.mkdir(parents=True, exist_ok=True)
    with TMP_ZONOS_JSON.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def synthesize_client_audio(client_text: str, turn: int) -> Path:
    """
    client_text -> directed_utterance JSON -> call synth_single_utterance in-process
    """
    # 1) text -> directed_utterance
    directed = make_directed_zonos_for_text(client_text)
    write_tmp_zonos_json(directed)

    # 2) เรียก Zonos โดยใช้ไฟล์ tmp JSON นี้
    print(f"[TURN {turn}] Calling Zonos synth (in-process)...")
    out_path_str = synth_single_utterance(turn, str(TMP_ZONOS_JSON))
    wav_path = Path(out_path_str)

    if not wav_path.exists():
        raise FileNotFoundError(f"Expected audio not found: {wav_path}")
    return wav_path

# ---------------------------------
# 3.2 WavLM SER: wav -> (val_s, aro_s)
# ---------------------------------

WAVLM_MODEL_NAME = "3loi/SER-Odyssey-Baseline-WavLM-Multi-Attributes"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading WavLM emotion model {WAVLM_MODEL_NAME} on {device} ...")
_wavlm = AutoModelForAudioClassification.from_pretrained(
    WAVLM_MODEL_NAME,
    trust_remote_code=True,
).to(device)
_wavlm.eval()

_target_sr = _wavlm.config.sampling_rate
_mean = _wavlm.config.mean
_std = _wavlm.config.std
_id2label = _wavlm.config.id2label  # {0: 'arousal', 1: 'dominance', 2: 'valence'}
print("WavLM id2label:", _id2label)


def _predict_file(path: str) -> Tuple[float, float, float]:
    """
    คืนค่า (aro, dom, val) ช่วงประมาณ 0..1 จากไฟล์เสียงเดียว
    """
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != _target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=_target_sr)
        sr = _target_sr

    audio = (audio - _mean) / (_std + 1e-6)

    wavs = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(device)
    mask = torch.ones(1, wavs.shape[1], dtype=torch.float32).to(device)

    with torch.no_grad():
        pred = _wavlm(wavs, mask)

    logits = pred.cpu().numpy()[0].astype(float)  # [A, D, V]
    aro = float(logits[0])
    dom = float(logits[1])
    val = float(logits[2])
    return aro, dom, val


def _scale_0_1_to_minus1_1(x: float) -> float:
    # ถ้า model ให้ 0..1, map ไป -1..1
    return 2.0 * x - 1.0


def get_speech_VA(wav_path: Path) -> Tuple[float, float]:
    """
    รับ path ของ wav แล้วคืน (val_s, aro_s) ในช่วง [-1, 1]
    """
    aro, dom, val = _predict_file(str(wav_path))
    val_s = _scale_0_1_to_minus1_1(val)
    aro_s = _scale_0_1_to_minus1_1(aro)
    return val_s, aro_s


Loading WavLM emotion model 3loi/SER-Odyssey-Baseline-WavLM-Multi-Attributes on cuda ...
WavLM id2label: {0: 'arousal', 1: 'dominance', 2: 'valence'}


## 4. System prompt client & therapist

In [7]:
CLIENT_SYSTEM = """
You are a CBT therapy client talking to therapist "Luna".

- You have anxiety, guilt, and loneliness related to your life.
- Speak in a natural, first-person voice.
- Stay emotionally consistent across turns.
- Describe thoughts, feelings, and situations in 2–4 sentences per turn.
"""

CLIENT_USER_TEMPLATE_FIRST = """
Start the first message to your therapist.
Describe what has been bothering you lately (2–4 sentences).
"""

CLIENT_USER_TEMPLATE_NEXT = """
Therapist just said:
"{therapist_text}"

Continue the conversation as the client.
Describe what you think and feel now in 2–4 sentences.
"""

THERAPIST_SYSTEM_DISS = """
You are "Luna", a CBT therapist with access to both the client's words and
an analysis of how their voice matches (or mismatches) those words.

For each client message you receive:
- Text-based emotion:
  - Valence_text, Arousal_text (from -1 to +1)
- Voice-based emotion:
  - Valence_speech, Arousal_speech (from -1 to +1)
- Dissonance:
  - delta_valence = speech - text
  - delta_arousal = speech - text

Interpretation guidelines:
- Large |delta_valence| or |delta_arousal| means the client's tone and words
  are pulling in different directions (they might be minimizing or masking something).
- Example: text seems "I'm fine" (positive) but voice is very flat or tense (negative).

Your job:
- When dissonance is small, respond as in normal emotion-aware CBT.
- When dissonance is large, gently explore the mismatch:
  - Reflect what might be "under the surface".
  - Ask curious, non-judgmental questions like
    "I wonder if part of you feels more scared/sad than the words suggest?"

Important:
- NEVER mention numbers, "dissonance", or "analysis".
- Speak only in natural language.
- Still follow CBT principles (thoughts, evidence, alternative perspectives).
"""

THERAPIST_USER_TEMPLATE_DISS = """
Client just said:
"{client_text}"

Estimates from analysis:
- Text emotion:
    - Valence_text: {val_t:.2f}
    - Arousal_text: {aro_t:.2f}
- Voice emotion:
    - Valence_speech: {val_s:.2f}
    - Arousal_speech: {aro_s:.2f}
- Dissonance (speech - text):
    - delta_valence: {delta_v:.2f}
    - delta_arousal: {delta_a:.2f}

Overall dissonance flag: {is_dissonant}

Write your next therapist response using this information internally.
If the mismatch (absolute delta) is large, gently explore what might be
unspoken or minimized, without naming any numbers.
"""



## 5. helper เรียก LLM

In [8]:
def chat_once(system_prompt: str, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

## 6. main loop – dialogue_6 dissonance-aware

In [9]:
import json
from pathlib import Path

def save_dialogue_json_and_jsonl(turns, base_path: str):
    """
    base_path เช่น 'dialogue_6_full_dissonance_online'
    จะได้:
      - dialogue_6_full_dissonance_online.json
      - dialogue_6_full_dissonance_online.jsonl
    """
    base = Path(base_path)
    json_path = base.with_suffix(".json")
    jsonl_path = base.with_suffix(".jsonl")

    # 1) เซฟแบบ JSON (list เต็ม ๆ สำหรับมนุษย์อ่าน)
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(turns, f, ensure_ascii=False, indent=2)
    print(f"[SAVE] JSON   -> {json_path}")

    # 2) เซฟแบบ JSONL (หนึ่ง turn ต่อ 1 บรรทัด สำหรับ LLM/สคริปต์อ่าน)
    with jsonl_path.open("w", encoding="utf-8") as f:
        for rec in turns:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[SAVE] JSONL  -> {jsonl_path}")

In [10]:
def run_dialogue_dissonance(
    max_turns: int = 10,
    out_path: str = "dialogue_1_full_dissonance_online.json",
):
    """
    1 turn = client พูด 1 ครั้ง + therapist ตอบ 1 ครั้ง
    """
    turns = []

    # ---- turn 1: client เริ่ม ----
    client_text = chat_once(CLIENT_SYSTEM, CLIENT_USER_TEMPLATE_FIRST)
    print(f"CLIENT (t=1): {client_text}\n")

    # 1) text VA
    val_t, aro_t = get_text_VA(client_text)

    # 2) สร้างเสียงจาก client text (ให้คุณไปเติมให้ gen wav จริง)
    wav_path = synthesize_client_audio(client_text, turn=1)

    # 3) speech VA จากไฟล์เสียง
    val_s, aro_s = get_speech_VA(wav_path)

    DISSONANCE_THRESHOLD = 0.5  # |delta| เกินค่านี้ = dissonance

    # 4) delta + flag dissonance
    delta_v = val_s - val_t
    delta_a = aro_s - aro_t
    is_dissonant = (abs(delta_v) >= DISSONANCE_THRESHOLD) or (abs(delta_a) >= DISSONANCE_THRESHOLD)

    print(f"[TURN 1] text VA    : val_t={val_t:.3f}, aro_t={aro_t:.3f}")
    print(f"[TURN 1] speech VA  : val_s={val_s:.3f}, aro_s={aro_s:.3f}")
    print(f"[TURN 1] dissonance : delta_v={delta_v:.3f}, delta_a={delta_a:.3f}, "
        f"is_dissonant={is_dissonant}\n")

    therapist_text = chat_once(
        THERAPIST_SYSTEM_DISS,
        THERAPIST_USER_TEMPLATE_DISS.format(
            client_text=client_text,
            val_t=val_t, aro_t=aro_t,
            val_s=val_s, aro_s=aro_s,
            delta_v=delta_v, delta_a=delta_a,
            is_dissonant=is_dissonant,  # ถ้าจะใช้ใน template ด้วย
        ),
    )
    print(f"THERAPIST (t=1): {therapist_text}\n")

    turns.append({
        "turn": 1,
        "client": client_text,
        "therapist": therapist_text,
        "condition": "dissonance_aware_therapist",
        "val_t": val_t, "aro_t": aro_t,
        "val_s": val_s, "aro_s": aro_s,
        "delta_valence": delta_v,
        "delta_arousal": delta_a,
        "is_dissonant": is_dissonant,
        "audio_path": str(wav_path),
    })


    # ---- turns 2..max_turns ----
    for t in range(2, max_turns + 1):
        client_text = chat_once(
            CLIENT_SYSTEM,
            CLIENT_USER_TEMPLATE_NEXT.format(therapist_text=therapist_text),
        )
        print(f"CLIENT (t={t}): {client_text}\n")

        val_t, aro_t = get_text_VA(client_text)
        wav_path = synthesize_client_audio(client_text, turn=t)
        val_s, aro_s = get_speech_VA(wav_path)
        delta_v = val_s - val_t
        delta_a = aro_s - aro_t
        is_dissonant = (abs(delta_v) >= DISSONANCE_THRESHOLD) or (abs(delta_a) >= DISSONANCE_THRESHOLD)

        print(f"[TURN {t}] text VA    : val_t={val_t:.3f}, aro_t={aro_t:.3f}")
        print(f"[TURN {t}] speech VA  : val_s={val_s:.3f}, aro_s={aro_s:.3f}")
        print(f"[TURN {t}] dissonance : delta_v={delta_v:.3f}, delta_a={delta_a:.3f}, "
            f"is_dissonant={is_dissonant}\n")

        therapist_text = chat_once(
            THERAPIST_SYSTEM_DISS,
            THERAPIST_USER_TEMPLATE_DISS.format(
                client_text=client_text,
                val_t=val_t, aro_t=aro_t,
                val_s=val_s, aro_s=aro_s,
                delta_v=delta_v, delta_a=delta_a,
                is_dissonant=is_dissonant,  # ถ้าจะใช้ใน template
            ),
        )
        print(f"THERAPIST (t={t}): {therapist_text}\n")

        turns.append({
            "turn": t,
            "client": client_text,
            "therapist": therapist_text,
            "condition": "dissonance_aware_therapist",
            "val_t": val_t, "aro_t": aro_t,
            "val_s": val_s, "aro_s": aro_s,
            "delta_valence": delta_v,
            "delta_arousal": delta_a,
            "is_dissonant": is_dissonant,
            "audio_path": str(wav_path),
        })

    # --- เซฟทั้ง .json และ .jsonl ---
    # ใช้ out_path เป็น base ชื่อไฟล์
    base_no_suffix = str(Path(out_path).with_suffix(""))
    save_dialogue_json_and_jsonl(turns, base_no_suffix)

if __name__ == "__main__":
    run_dialogue_dissonance(max_turns=10)


CLIENT (t=1): Hi Luna, I've been feeling really overwhelmed lately. My anxiety seems to be getting worse, especially when I think about my future and the decisions I have to make. I often find myself caught up in guilt about not doing enough, which just adds to my loneliness. It feels like I'm stuck in this cycle where I can't escape my thoughts.

[TURN 1] Calling Zonos synth (in-process)...
Using INPUT_JSON: C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\tmp_directed_zonos_single.json
Total utterances in JSON: 1
Expected minimum duration ~11.69s for utterance 1
[Zonos] Utterance 1 attempt 1/3


Generating: 100%|██████████| 2588/2588 [01:54<00:00, 22.68it/s]


Attempt 1: duration=29.95s, rms=0.000
[Zonos] Utterance 1 attempt 2/3


Generating:  86%|████████▌ | 2214/2588 [01:30<00:15, 24.34it/s]


Attempt 2: duration=25.58s, rms=0.187
[Zonos] Utterance 1 attempt 3/3


...

Attempt 1: duration=29.95s, rms=0.070
[Zonos] Utterance 10 attempt 2/3


Generating: 100%|██████████| 2588/2588 [02:02<00:00, 21.21it/s]


Attempt 2: duration=29.95s, rms=0.000
[Zonos] Utterance 10 attempt 3/3


Generating: 100%|██████████| 2588/2588 [02:01<00:00, 21.25it/s]


Attempt 3: duration=1.83s, rms=0.112
[FALLBACK] Saved best-effort audio for utterance 10 (dur=29.95s, rms=0.070)
[TURN 10] text VA    : val_t=-0.136, aro_t=0.198
[TURN 10] speech VA  : val_s=-0.173, aro_s=0.166
[TURN 10] dissonance : delta_v=-0.037, delta_a=-0.032, is_dissonant=False

THERAPIST (t=10): It sounds like you’re really navigating a lot of tension and conflicting feelings around connection right now. It’s understandable to crave that connection while also feeling fearful of getting hurt again. That guilt you mentioned about not trying harder is something many people can relate to, but it’s important to recognize that your feelings are valid. 

You’ve expressed a desire for reassurance and for strategies to help you open up gradually, which shows you’re already taking steps toward addressing these feelings. What are some small steps you think you could take that might feel manageable for you? And I wonder if part of you feels more anxious or hesitant about reaching out than y